# Aprendizado Contrastivo

Um classificador escolhe entre classes fixas, definidas no treino. Em muitas aplicações as classes não são conhecidas de antemão: um sistema de reconhecimento facial precisa reconhecer funcionários contratados depois do treino, e um verificador de assinaturas precisa funcionar para clientes novos. Retreinar a rede a cada pessoa nova não é viável, e muitas vezes há uma única foto ou assinatura de referência.

Aprender classes novas a partir de poucos exemplos é o problema do **few-shot learning**. Treinar um classificador só com esses exemplos não funciona: com uma ou cinco imagens por classe, a rede decora as imagens e não generaliza.

O **aprendizado contrastivo** troca a pergunta "qual é a classe?" por "esses dois exemplos são a mesma coisa?". A rede transforma cada imagem em um vetor, o **embedding**, e é treinada, com muitas classes e muitos exemplos, para que exemplos da mesma classe fiquem próximos e exemplos de classes diferentes fiquem distantes. Para reconhecer uma classe nova, basta comparar o embedding de uma imagem com os dos poucos exemplos dela, sem treinar nada.

Vamos treinar uma CNN pequena no Omniglot e testá-la em caracteres de alfabetos que ela nunca viu, com um ou cinco exemplos de cada.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import Omniglot

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## O dataset

O **Omniglot** tem 1623 caracteres de 50 alfabetos, cada um desenhado por 20 pessoas diferentes: muitas classes com poucos exemplos cada, o contrário do MNIST. O `torchvision` o divide em dois conjuntos de alfabetos distintos: o `background`, com 964 caracteres, para o treino, e o de avaliação, com 659 caracteres de outros alfabetos. Testar no segundo mede se a rede aprendeu a comparar caracteres em geral, e não apenas os do treino.

As imagens têm $105 \times 105$ pixels, com o traço preto sobre o fundo branco. Reduzimos para $28 \times 28$ e invertemos as cores, para o traço valer 1 e o fundo 0, como no MNIST. As 20 imagens de cada caractere vêm em sequência, e guardamos tudo em um tensor `[caracteres, 20, 1, 28, 28]`.

In [ ]:
IMAGE_SIZE = 28
transform = transforms.Compose([transforms.Resize(IMAGE_SIZE), transforms.ToTensor()])


def load_omniglot(background):
    dataset = Omniglot(root="data", background=background, download=True, transform=transform)
    images = torch.stack([1 - image for image, _ in dataset])   # inverte: traço = 1, fundo = 0
    return images.reshape(-1, 20, 1, IMAGE_SIZE, IMAGE_SIZE)


train_images = load_omniglot(background=True)
test_images = load_omniglot(background=False)
print(train_images.shape, test_images.shape)

Na primeira linha, dez caracteres diferentes; na segunda, dez das 20 versões de um mesmo caractere. Cada pessoa desenha com uma espessura, inclinação e proporção próprias, e a rede precisa reconhecer o caractere apesar disso.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for k in range(10):
    axes[0, k].imshow(train_images[100 * k, 0, 0], cmap="gray")
    axes[1, k].imshow(train_images[0, k, 0], cmap="gray")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

## Pares positivos

O treino usa **pares positivos**: duas imagens do mesmo caractere, desenhadas por pessoas diferentes. O `PairDataset` tem um item por caractere e, a cada chamada, sorteia duas das 20 imagens dele. Com `shuffle=True`, cada lote tem caracteres diferentes, e os outros caracteres do lote servem como **negativos**: a rede precisa aproximar os dois desenhos de cada par e afastá-los dos demais. Não é preciso montar os pares negativos à mão.

In [ ]:
class PairDataset(Dataset):
    def __init__(self, images):
        self.images = images   # [caracteres, 20, 1, 28, 28]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, c):
        i, j = torch.randperm(20)[:2]   # duas versões diferentes do caractere c
        return self.images[c, i], self.images[c, j]


train_dataloader = DataLoader(PairDataset(train_images), batch_size=128, shuffle=True, drop_last=True)

anchors, positives = next(iter(train_dataloader))
print(anchors.shape, positives.shape)   # [128, 1, 28, 28] cada

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for k in range(8):
    axes[0, k].imshow(anchors[k, 0], cmap="gray")
    axes[1, k].imshow(positives[k, 0], cmap="gray")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

## A rede

O **encoder** é uma CNN como as dos classificadores, com três blocos de convolução, e termina em uma camada linear com 64 saídas no lugar das classes. Uma imagem `[1, 28, 28]` vira um vetor de 64 números, o embedding.

O embedding é normalizado para ter norma 1, e todos os vetores ficam sobre uma esfera. Assim, o produto interno entre dois embeddings é a **similaridade do cosseno**, que vai de $-1$, vetores opostos, a $1$, vetores iguais:

$$\text{sim}(z, z') = z^\top z' = \cos \theta$$

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class Encoder(nn.Module):
    def __init__(self, embedding_dim=64):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 32),    # [N, 32, 14, 14]
            conv_block(32, 64),   # [N, 64, 7, 7]
            conv_block(64, 128),  # [N, 128, 3, 3]
            nn.Flatten(),         # [N, 1152]
        )
        self.fc = nn.Linear(128 * 3 * 3, embedding_dim)

    def forward(self, x):
        z = self.fc(self.features(x))   # [N, 64]
        return F.normalize(z, dim=1)    # norma 1


model = Encoder()
print(model(anchors).shape)
print(f"{sum(p.numel() for p in model.parameters()):,} parâmetros")

## A perda InfoNCE

Em um lote de $N$ pares, sejam $z_i$ o embedding da primeira imagem do par $i$ e $z'_j$ o da segunda imagem do par $j$. A matriz de similaridades $S_{ij} = z_i^\top z'_j / \tau$ tem os pares positivos na diagonal e os negativos fora dela. Cada linha pode ser lida como uma classificação com $N$ opções: entre as $N$ segundas imagens, qual é o par da imagem $i$? A resposta certa é a coluna $i$, e a perda é a entropia cruzada de cada linha:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(S_{ii})}{\sum_{j=1}^{N} \exp(S_{ij})}$$

Esta é a perda **InfoNCE**, usada no SimCLR e no CLIP. Para diminuí-la, a rede precisa aumentar a similaridade do par positivo e diminuir a dos $N - 1$ negativos. Aplicamos a mesma perda às colunas, que perguntam o inverso, e tiramos a média das duas.

A **temperatura** $\tau$ ajusta a escala. Como a similaridade do cosseno fica entre $-1$ e $1$, sem a divisão a softmax ficaria quase uniforme mesmo com a rede certa. Com $\tau = 0{,}1$ as similaridades vão de $-10$ a $10$, e a rede consegue dar uma probabilidade alta ao par certo.

No código, a matriz é um único produto de matrizes, e o alvo de cada linha é o seu próprio índice, `[0, 1, ..., N-1]`.

In [ ]:
def info_nce(z1, z2, temperature=0.1):
    logits = z1 @ z2.T / temperature                  # [N, N], positivos na diagonal
    targets = torch.arange(len(z1), device=z1.device)
    return (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2

## Treinamento

O laço é o dos classificadores, com as duas imagens de cada par passando pela mesma rede. Uma época percorre os 964 caracteres uma vez, apenas 7 lotes, por isso treinamos por 100 épocas.

In [ ]:
def train(model, train_dataloader, num_epochs, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for anchors, positives in train_dataloader:
            anchors, positives = anchors.to(device), positives.to(device)
            loss = info_nce(model(anchors), model(positives))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        losses.append(total_loss / len(train_dataloader))
        if (epoch + 1) % 10 == 0:
            print(f"[{epoch + 1}/{num_epochs}] perda {losses[-1]:.3f}")

    return losses

In [ ]:
torch.manual_seed(0)
untrained = Encoder().to(device)   # referência: a mesma rede sem treino
model = Encoder().to(device)
losses = train(model, train_dataloader, num_epochs=100)

plt.plot(range(1, len(losses) + 1), losses)
plt.title("perda de treino")
plt.xlabel("época")
plt.ylabel("InfoNCE")
plt.show()

## Avaliação em caracteres inéditos

A partir daqui usamos só o conjunto de avaliação, com alfabetos que a rede nunca viu. A `embed` calcula o embedding das 20 imagens de cada caractere, em um tensor `[659, 20, 64]`.

Primeiro, a matriz de similaridades entre duas versões de 12 caracteres. Na rede sem treino, tudo se parece com tudo: caracteres são traços finos sobre um fundo vazio. Depois do treino, a diagonal se destaca.

In [ ]:
def embed(model, images):
    model.eval()
    batches = images.reshape(-1, 1, IMAGE_SIZE, IMAGE_SIZE).split(1000)   # em blocos, para caber na memória
    with torch.no_grad():
        z = torch.cat([model(batch.to(device)).cpu() for batch in batches])
    return z.reshape(len(images), 20, -1)   # [caracteres, 20, 64]


test_untrained = embed(untrained, test_images)
test_embeddings = embed(model, test_images)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, z, title in zip(axes, [test_untrained, test_embeddings], ["sem treino", "treinada"]):
    similarity = z[:12, 0] @ z[:12, 1].T   # versão 0 contra versão 1 dos 12 primeiros caracteres
    image = ax.imshow(similarity, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_title(title)
    ax.set_xlabel("versão 1")
    ax.set_ylabel("versão 0")
fig.colorbar(image, ax=axes)
plt.show()

### Classificação com poucos exemplos

O few-shot learning é avaliado com tarefas **N-way K-shot**: sorteamos $N$ caracteres e damos $K$ exemplos de cada, o **suporte**. Para uma imagem nova de um desses caracteres, a **consulta**, a rede responde o caractere mais similar. Ao acaso, a acurácia seria $1/N$: 20% com 5 caracteres e 5% com 20. Com $K = 1$, a tarefa é chamada de **one-shot**.

Com mais de um exemplo, cada caractere é representado pela média dos seus $K$ embeddings, normalizada, o **protótipo**. A média cancela parte das variações de cada desenho, como a espessura e a inclinação, e fica mais perto do "caractere ideal" que qualquer exemplo isolado. É a ideia das *Prototypical Networks* (Snell et al., 2017). Com $K = 1$, o protótipo é o próprio exemplo.

A `few_shot_accuracy` repete a tarefa 1000 vezes, sorteando os caracteres e as imagens de cada vez. A consulta é sempre uma versão que não está no suporte.

In [ ]:
def few_shot_accuracy(embeddings, n_way, k_shot, episodes=1000):
    generator = torch.Generator().manual_seed(0)
    correct = 0
    for _ in range(episodes):
        characters = torch.randperm(len(embeddings), generator=generator)[:n_way]
        versions = torch.randperm(20, generator=generator)
        support = embeddings[characters][:, versions[:k_shot]]   # [n_way, k_shot, 64]
        prototypes = F.normalize(support.mean(dim=1), dim=1)     # [n_way, 64]
        query = embeddings[characters, versions[k_shot]]         # [n_way, 64], fora do suporte
        predicted = (query @ prototypes.T).argmax(dim=1)
        correct += (predicted == torch.arange(n_way)).float().mean().item()
    return correct / episodes


for n_way in [5, 20]:
    for k_shot in [1, 5]:
        print(f"{n_way}-way {k_shot}-shot: sem treino {few_shot_accuracy(test_untrained, n_way, k_shot):.1%}, "
              f"treinada {few_shot_accuracy(test_embeddings, n_way, k_shot):.1%}")

Com cinco exemplos, a rede treinada erra cerca de três vezes menos que com um, sem nenhum treino a mais: só muda a referência de cada caractere. A rede sem treino também melhora com o protótipo, mas continua longe, porque a média não corrige embeddings que não separam os caracteres. Um cadastro com várias fotos por pessoa, comparadas com a média delas, é o mesmo protótipo aplicado ao reconhecimento facial.

Uma tarefa 5-way 1-shot em imagens: a consulta à esquerda e os cinco suportes com a similaridade de cada um, em verde o caractere certo.

In [ ]:
characters = torch.tensor([3, 50, 150, 300, 500])
query = test_embeddings[characters[0], 5]
similarities = test_embeddings[characters, 0] @ query

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
axes[0].imshow(test_images[characters[0], 5, 0], cmap="gray")
axes[0].set_title("consulta")
for k, ax in enumerate(axes[1:]):
    ax.imshow(test_images[characters[k], 0, 0], cmap="gray")
    ax.set_title(f"sim {similarities[k]:.2f}", color="green" if k == 0 else "red")
for ax in axes:
    ax.axis("off")
plt.show()

### Verificação

Reconhecimento facial e verificação de assinaturas costumam ser **verificação**: não há uma lista de candidatos, apenas uma referência e a pergunta "é a mesma pessoa?". A resposta vem de um **limiar**: se a similaridade passa dele, é a mesma.

Comparamos cada imagem com outra versão do mesmo caractere e com a versão de um caractere diferente, e vemos as duas distribuições de similaridade. O limiar é escolhido nos caracteres de treino, maximizando a acurácia, e aplicado aos de avaliação.

In [ ]:
def pair_similarities(embeddings):
    same = (embeddings[:, 0] * embeddings[:, 1]).sum(dim=1)                       # mesmo caractere
    different = (embeddings[:, 0] * embeddings.roll(1, dims=0)[:, 1]).sum(dim=1)  # caractere vizinho
    return same, different


def verification_accuracy(same, different, threshold):
    return ((same > threshold).sum() + (different <= threshold).sum()).item() / (len(same) + len(different))


train_same, train_different = pair_similarities(embed(model, train_images))
thresholds = torch.linspace(-1, 1, 201)
threshold = max(thresholds, key=lambda t: verification_accuracy(train_same, train_different, t))

same, different = pair_similarities(test_embeddings)
print(f"limiar {threshold:.2f}, acurácia {verification_accuracy(same, different, threshold):.1%}")

plt.hist(same, bins=40, alpha=0.6, label="mesmo caractere")
plt.hist(different, bins=40, alpha=0.6, label="caracteres diferentes")
plt.axvline(threshold, color="black", linestyle="--", label="limiar")
plt.xlabel("similaridade do cosseno")
plt.legend()
plt.show()

A acurácia fica abaixo da classificação 5-way porque o caractere diferente é o vizinho na lista, quase sempre do mesmo alfabeto e parecido com o original. Os erros ficam na região em que as duas distribuições se encontram, e mover o limiar troca um tipo de erro pelo outro: um limiar alto recusa mais versões legítimas, e um baixo aceita mais impostores. Um sistema de controle de acesso prefere errar recusando; um álbum de fotos que agrupa rostos prefere errar juntando.

## Exercícios

### Exercício 1: temperatura

Treine com `temperature` igual a 0,01, 0,1 e 1 na `info_nce` e compare a acurácia 20-way 1-shot. O que acontece com as duas distribuições do histograma de verificação quando $\tau$ é grande?

### Exercício 2: sem rótulos

O treino usou o rótulo para formar os pares: duas imagens do mesmo caractere. O **SimCLR** forma o par sem rótulos, aplicando duas transformações aleatórias à **mesma** imagem. Mude o `PairDataset` para sortear uma imagem qualquer e devolver duas versões dela com `transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2))`. Quanto a acurácia cai sem os rótulos? Que transformações ajudam?

### Exercício 3: dimensão do embedding

Treine com `embedding_dim=2`. Os embeddings normalizados ficam sobre um círculo: desenhe, em um gráfico de dispersão, as 20 versões de 10 caracteres de avaliação, uma cor por caractere. Depois aumente a dimensão para 8, 16 e 128 e compare a acurácia 20-way 1-shot. A partir de que dimensão ela para de melhorar?